In [ ]:
import numpy as np
from scipy.stats import binom
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# --- GLOBAL INITIALIZATION & CONFIGURATION ---
n_users = 35         # Secondary users (Dimensions D)
n_primary = 8        # Primary users
n_channels = 40      # Available spectrum bands (Search Space S = n_channels - 1)
noise = 0.1          # Noise floor for communication SNR

PS_penalty = 75      # Penalty for primary-secondary user channel collision
SS_penalty = 25      # Penalty for secondary-secondary user channel collision

N = 200              # Number of particles
S = n_channels - 1   # Search space boundary [0, n_channels-1]
D = n_users          # Dimensions = number of users
n_iter = 600         # Number of updates of positions
s = 5                # Random seed for algorithms

# Real-time environment update frequency
ENV_UPDATE_INTERVAL = 100 

# PSO Optimization Parameters
a = 0.9
b = 1.496
b_hat = 1.496
c = 0.5

# QPSO Optimization Parameters
beta_start = 0.25
beta_end = 0.65

# --- QUANTUM ILLUMINATION SENSING CONFIGURATION ---
N_WINDOW = 40             # Number of entangled photon pairs sent per channel sensing cycle
TARGET_PFA = 0.05         # Target False Alarm rate bound
SU_SENSING_SNR_DB = 4.0   # Sensing SNR if a PU is physically present

_BACKEND = AerSimulator()

# --- ENVIRONMENT SEEDING & SETUP ---
np.random.seed(s)

# Generate channel gains and calculate standard communication SNR
channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels))
SNR = channel_gain / noise

# --- DYNAMIC ENVIRONMENT STATE TARGETS ---
# Global runtime variables tracked and changed dynamically
PU_occupied = np.zeros(n_channels, dtype=int)
observed_spectrum_map = np.zeros(n_channels, dtype=int)


# --- QUANTUM ILLUMINATION FUNCTIONS ---
def snr_db_to_ploss(snr_db, floor=0.02, ceiling=0.98):
    """Maps the operating sensing SNR to a depolarizing channel error probability."""
    gamma = 10 ** (snr_db / 10.0)
    p_loss = 1.0 / (1.0 + gamma)
    return float(np.clip(p_loss, floor, ceiling))

def _quantum_circuit():
    """Builds a 2-qubit circuit implementing the Quantum Illumination protocol."""
    qc = QuantumCircuit(2, 2)
    qc.h(0)        # Idler photon
    qc.cx(0, 1)    # Entangle idler (0) with signal (1)
    qc.id(1)       # Environmental channel interaction placeholder
    qc.cx(0, 1)    # Quantum receiver decoding step
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc

def calculate_decision_threshold(p0, n_window, target_pfa):
    """Computes the optimal statistical hit threshold (tau) using Neyman-Pearson logic."""
    tau = n_window + 1
    for t in range(n_window + 1):
        pfa_t = 1.0 if t == 0 else binom.sf(t - 1, n_window, p0)
        if pfa_t <= target_pfa:
            tau = t
            break
    return tau

def update_pu_occupancy():
    """Dynamically alters the ground-truth channel positions of active PUs."""
    global PU_occupied
    PU_occupied = np.zeros(n_channels, dtype=int)
    PU_occupied[np.random.choice(n_channels, n_primary, replace=False)] = 1

def perform_qi_sensing_live(p0, tau, qc_compiled):
    """
    Performs a real-time Quantum Illumination spectrum sweep across all channels.
    Modifies the global observed_spectrum_map array directly.
    """
    global observed_spectrum_map
    p_loss = snr_db_to_ploss(SU_SENSING_SNR_DB)
    new_map = np.zeros(n_channels, dtype=int)
    
    for ch in range(n_channels):
        p_dep = p_loss if PU_occupied[ch] == 1 else 1.0
        nm = NoiseModel()
        nm.add_quantum_error(depolarizing_error(p_dep, 1), "id", [1])
        
        job = _BACKEND.run(qc_compiled, shots=N_WINDOW, noise_model=nm)
        hits = job.result().get_counts().get("00", 0)
        
        new_map[ch] = 1 if hits >= tau else 0
        
    observed_spectrum_map = new_map


# --- DYNAMIC COGNITIVE FITNESS FUNCTION ---
def dynamic_fitness(x):
    """Evaluates solutions using the real-time runtime state of the QI sensor map."""
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    
    # Particles read the *current* real-time state of the QI sensory map
    sensed_pu_mask = observed_spectrum_map[channels]
    
    sinr_vals = SNR[np.arange(n_users), channels]
    throughput = np.sum(np.log2(1 + sinr_vals) * (1 - sensed_pu_mask))
    ps_penalty = np.sum(sensed_pu_mask) * PS_penalty
    
    counts = np.bincount(channels, minlength=n_channels)
    ss_pen = np.sum(np.maximum(counts - 1, 0)) * SS_penalty
    
    return -throughput + ps_penalty + ss_pen


# --- OPTIMIZATION ALGORITHMS WITH CONTEXT SHIFT TRACKING ---
def dpso_dynamic(f, D, N, S, n_iter, a, b, b_hat, c, see, p0, tau, qc_compiled):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))
    v = np.random.normal(size=(N, D))
    p = x.copy()
    fp = np.array([f(p[i]) for i in range(N)])
    p_hat = x[np.argmin(fp)].copy()
    fp_hat = f(p_hat)
    fp_hat_his = []

    for i in range(n_iter):
        # REAL-TIME TRIGGER: Shift environment and resense at regular intervals
        if i % ENV_UPDATE_INTERVAL == 0 and i > 0:
            update_pu_occupancy()
            perform_qi_sensing_live(p0, tau, qc_compiled)
            # CRITICAL: Re-evaluate swarm memory based on new real-time reality
            fp = np.array([f(p[k]) for k in range(N)])
            p_hat = p[np.argmin(fp)].copy()
            fp_hat = f(p_hat)

        fp_hat_his.append(float(fp_hat))
        r, r_hat = np.random.uniform(0, 1, (2, N, D))
        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)
        x = x + c*v
        x = np.clip(x, 0, S)

        for n in range(N):
            xn = x[n]
            fxn = f(xn)
            if fxn < fp[n]:
                p[n] = xn.copy()
                fp[n] = fxn
                if fxn < fp_hat:
                    p_hat = xn.copy()
                    fp_hat = fxn
                    
    return p_hat, fp_hat_his

def dqpso_dynamic(f, D, N, S, n_iter, beta_start, beta_end, see, p0, tau, qc_compiled):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))
    p = x.copy()
    fp = np.array([f(p[i]) for i in range(N)])
    p_hat = x[np.argmin(fp)].copy()
    fp_hat = f(p_hat)
    fp_hat_his = []

    for i in range(n_iter):
        # REAL-TIME TRIGGER: Shift environment and resense at regular intervals
        if i % ENV_UPDATE_INTERVAL == 0 and i > 0:
            update_pu_occupancy()
            perform_qi_sensing_live(p0, tau, qc_compiled)
            # CRITICAL: Re-evaluate swarm memory based on new real-time reality
            fp = np.array([f(p[k]) for k in range(N)])
            p_hat = p[np.argmin(fp)].copy()
            fp_hat = f(p_hat)

        fp_hat_his.append(float(fp_hat))
        beta = beta_start - (beta_start - beta_end) * i / n_iter
        mbest = np.mean(p, axis=0)

        phi = np.random.uniform(0, 1, (N, D))
        p_local = phi * p + (1 - phi) * p_hat

        u = np.random.uniform(1e-12, 1, (N, D))
        sign = 2 * np.random.randint(0, 2, size=(N, D)) - 1
        x = p_local + sign * beta * np.abs(mbest - x) * np.log(1/u)
        x = np.clip(x, 0, S)

        for n in range(N):
            xn = x[n]
            fxn = f(xn)
            if fxn < fp[n]:
                p[n] = xn.copy()
                fp[n] = fxn
                if fxn < fp_hat:
                    p_hat = xn.copy()
                    fp_hat = fxn
                    
    return p_hat, fp_hat_his


# --- MAIN PIPELINE EXECUTION ---
if __name__ == "__main__":
    print("[1/3] Initializing and Calibrating Quantum Illumination Hardware Components...")
    # Setup quantum hardware primitives once to maximize execution efficiency
    qc_base = _quantum_circuit()
    qc_compiled = transpile(qc_base, basis_gates=["h", "cx", "id"], optimization_level=0)
    
    # Calibrate receiver decision variables under empty thermal channel conditions (H0)
    nm_h0 = NoiseModel()
    nm_h0.add_quantum_error(depolarizing_error(1.0, 1), "id", [1])
    cal_job = _BACKEND.run(qc_compiled, shots=2000, noise_model=nm_h0)
    p0 = cal_job.result().get_counts().get("00", 0) / 2000
    tau = calculate_decision_threshold(p0, N_WINDOW, TARGET_PFA)
    
    # Establish baseline initialization environment state
    update_pu_occupancy()
    perform_qi_sensing_live(p0, tau, qc_compiled)
    
    print(f"[2/3] Processing Classical DPSO Optimization Engine over {n_iter} Dynamic Ticks...")
    result_dpso, _ = dpso_dynamic(dynamic_fitness, D, N, S, n_iter, a, b, b_hat, c, s, p0, tau, qc_compiled)
    C_best_assignment = np.clip(np.round(result_dpso).astype(int), 0, n_channels - 1)
    
    # Store dynamic state post-DPSO execution to compute final realized metrics safely
    dpso_final_pu_truth = PU_occupied.copy()
    
    # Reset tracking environment state back to seed layout baseline for fair QPSO comparison
    np.random.seed(s)
    update_pu_occupancy()
    perform_qi_sensing_live(p0, tau, qc_compiled)
    
    print(f"[3/3] Processing Quantum-behaved QPSO Optimization Engine over {n_iter} Dynamic Ticks...")
    result_qpso, _ = dqpso_dynamic(dynamic_fitness, D, N, S, n_iter, beta_start, beta_end, s, p0, tau, qc_compiled)
    Q_best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels - 1)
    
    qpso_final_pu_truth = PU_occupied.copy()
    
    # --- CALCULATE LOGICAL PERFORMANCE MEASUREMENTS ---
    C_throughput = sum(np.log2(1 + SNR[u, C_best_assignment[u]]) for u in range(n_users) if not dpso_final_pu_truth[C_best_assignment[u]])
    Q_throughput = sum(np.log2(1 + SNR[u, Q_best_assignment[u]]) for u in range(n_users) if not qpso_final_pu_truth[Q_best_assignment[u]])
            
    print("\n" + "=" * 80)
    print("                DYNAMIC SPECTRUM TRACKING PERFORMANCE REPORT")
    print("=" * 80)
    print("DPSO Final Loop PU Ground Truth: ", dpso_final_pu_truth)
    print("DPSO Final Selected Profile:     ", C_best_assignment)
    print(f"Classical Realized Throughput:    {C_throughput:.4f} bps/Hz")
    print("-" * 80)
    print("QPSO Final Loop PU Ground Truth: ", qpso_final_pu_truth)
    print("QPSO Final Selected Profile:     ", Q_best_assignment)
    print(f"Quantum-behaved Throughput:       {Q_throughput:.4f} bps/Hz")
    print("=" * 80)

[1/3] Initializing and Calibrating Quantum Illumination Hardware Components...
[2/3] Processing Classical DPSO Optimization Engine over 600 Dynamic Ticks...
[3/3] Processing Quantum-behaved QPSO Optimization Engine over 600 Dynamic Ticks...

                DYNAMIC SPECTRUM TRACKING PERFORMANCE REPORT
DPSO Final Loop PU Ground Truth:  [0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 0
 0 0 0]
DPSO Final Selected Profile:      [39 39 31 30 30  2 39  1 37 36 10  6 23 26 29 13 39  4  0  8 18 32  7 25
  3 26 17 16  8 21 35 38 15 19 22]
Classical Realized Throughput:    98.0718 bps/Hz
--------------------------------------------------------------------------------
QPSO Final Loop PU Ground Truth:  [0 0 0 0 1 1 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 0 0 0
 0 0 0]
QPSO Final Selected Profile:      [ 3 27 28  9 35 31 28 34 16  6 34  8 32 15 12 11  2 36 38 28 16 26 24 23
 20  7  9 22 17 10 29 35 32 13 36]
Quantum-behaved Throughput:       96.9856 bps/Hz
